## ViT(Vision Transformer) 모델과 데이터셋을 활용하여 LoRA를 적용하여 fine-tuning하기

In [1]:
# 모델
# ViT 모델은 ImageNet 데이터로 사전학습된 모델
import torchvision.models as models
model = models.vit_b_16(weights="IMAGENET1K_V1")

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:05<00:00, 66.0MB/s]


In [2]:
# 데이터
# Oxford-IIIT Pet 데이터는 ImageNet에 포함된 강아지와 고양이로 구성되어있지만 class가 품종으로 구성
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.OxfordIIITPet(root='./data', split='trainval', target_types='category', transform=transform, download=True)
test_dataset = datasets.OxfordIIITPet(root='./data', split='test', target_types='category', transform=transform, download=True)


100%|██████████| 792M/792M [01:01<00:00, 12.8MB/s]  
100%|██████████| 19.2M/19.2M [00:02<00:00, 9.53MB/s]


In [3]:
# LoRA (Low-Rank Adaptation) 구현
import math
import torch
import torch.nn as nn
import torch.nn.utils.parametrize as parametrize

class LoRALayer(nn.Module):
    def __init__(self, out_dim, in_dim, r=8, alpha=16):
        super().__init__()
        self.lora_A = nn.Parameter(torch.empty(r, in_dim))
        self.lora_B = nn.Parameter(torch.zeros(out_dim, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scaling = alpha / r

    def forward(self, W):
        return W + (self.lora_B @ self.lora_A) * self.scaling # W' = W + (alpha / r) * B @ A

In [4]:
# 모델에 LoRA 적용
r, alpha = 8, 16

# 1) 사전학습된 파라미터 전체 동결
for p in model.parameters():
    p.requires_grad = False

# 2) 각 인코더 블록의 attention QKV projection(in_proj_weight)에 LoRA 주입
#    torchvision ViT는 Q, K, V weight가 하나로 합쳐진 in_proj_weight(2304x768)를 사용하므로
#    parametrize로 weight 자체에 W + BA를 덧붙입니다.
for block in model.encoder.layers:
    attn = block.self_attention
    out_dim, in_dim = attn.in_proj_weight.shape
    parametrize.register_parametrization(
        attn, "in_proj_weight", LoRALayer(out_dim, in_dim, r=r, alpha=alpha)
    )

# 3) 분류 head 교체: ImageNet 1000 클래스 -> Pet 37 품종 (새 head는 학습 대상)
num_classes = len(train_data.classes)
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

# 학습되는 파라미터 비율 확인
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

trainable params: 323,365 / 86,122,021 (0.38%)


In [5]:
# 학습 (fine-tuning)
# LoRA 파라미터와 새 분류 head만 학습됩니다. (나머지는 동결 상태)
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3
)

epochs = 3
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}/{epochs} | loss: {running_loss/total:.4f} | train acc: {correct/total:.4f}")

Epoch 1/3 | loss: 1.0448 | train acc: 0.8054
Epoch 2/3 | loss: 0.1593 | train acc: 0.9568
Epoch 3/3 | loss: 0.0699 | train acc: 0.9851


In [6]:
# 평가
# 학습이 끝난 LoRA 모델을 test set으로 평가합니다.
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {correct / total:.4f} ({correct}/{total})")

Test Accuracy: 0.9302 (3413/3669)
